# Experiment: 2D cond 1D

dim(x)=1, dim(y)=1 — comparing LGD vs LGD-CM.

In [ ]:
# ============================================================
# CONFIG — only this cell changes between notebooks
# Structure:
#   simulations/src/        ← all .py modules
#   simulations/notebooks/  ← this notebook
#   simulations/params/     ← canonical GMM parameters (shared, load first)
#   simulations/checkpoints/
#   simulations/results/
# ============================================================
EXPERIMENT_NAME   = "2D_cond_1D"
GLOBAL_SEED       = 42
FORCE_RETRAIN     = False

BASE_DIR          = "/content/conditional-matching-paper/simulations"
PARAMS_DIR        = f"{BASE_DIR}/params"
CHECKPOINT_DIR    = f"{BASE_DIR}/checkpoints/{EXPERIMENT_NAME}"
RESULTS_DIR       = f"{BASE_DIR}/results/{EXPERIMENT_NAME}"

# Architecture — Diffusion models
NBLOCKS           = 3
NUNITS            = 128

# Architecture — Consistency Model (separate so it can be scaled independently)
NBLOCKS_CM        = 3
NUNITS_CM         = 128

# Training — Diffusion
NEPOCHS           = 20_000
BATCH_SIZE        = 1_024

# Training — Consistency Model
NEPOCHS_CM        = 20_000
BATCH_SIZE_CM     = 1_024

# Diffusion
DIFFUSION_STEPS   = 100

# Optimization
N_ATTEMP_OPTIM              = 25
NSAMPLES_IN_OPTIM_FOR_MMD   = 250
NUM_X_T_LGD                 = 3
NUM_X_T_LGD_CM              = 3

# GMM dimensions
CONDITION_ON      = 1   # dim(x)=1, dim(y)=1

In [ ]:
!pip install flow_matching -q
!pip install POT -q

In [ ]:
import os, sys
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass
    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    token = github_token if github_token else getpass.getpass("Enter your GitHub personal access token: ")

    repo_url  = f"https://{token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "adding-simu-compare"

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

# ── point Python at simulations/src where all .py modules live ──
src_path = f"/content/{repo_name}/simulations/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Branch: {branch}")
print(f"src path on sys.path: {src_path}")

In [ ]:
import os, sys, time, json
import importlib
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
from tqdm import trange

import Diffusion
import LossFunctions
import ConsistencyModels
import dist_utils
import Optimization
import experiment_utils
from ConsistencyModels import ConsistencyModeliCT
import evalModels
from LossFunctions import MMDLoss, RBF

for mod in [Diffusion, LossFunctions, ConsistencyModels,
            dist_utils, Optimization, experiment_utils]:
    importlib.reload(mod)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)
os.makedirs(PARAMS_DIR,     exist_ok=True)
print("Imports done.")

In [ ]:
env_info = experiment_utils.get_environment_info()
experiment_utils.print_environment_info(env_info)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## GMM Parameters

In [ ]:
# ============================================================
# GMM PARAMETERS
# Priority:
#   1. Load from PARAMS_DIR  (shared across runs, committed to repo)
#   2. Load from RESULTS_DIR (fallback from a previous run)
#   3. Generate fresh and save to both dirs
#
# To force regeneration: set FORCE_REGENERATE_PARAMS = True
# ============================================================
FORCE_REGENERATE_PARAMS = False

def _load_params():
    """Try PARAMS_DIR first, then RESULTS_DIR."""
    loaded = experiment_utils.load_gmm_params(PARAMS_DIR, EXPERIMENT_NAME)
    if loaded is not None:
        print(f"[GMM] Loaded from PARAMS_DIR: {PARAMS_DIR}")
        return loaded
    loaded = experiment_utils.load_gmm_params(RESULTS_DIR, EXPERIMENT_NAME)
    if loaded is not None:
        print(f"[GMM] Loaded from RESULTS_DIR: {RESULTS_DIR}")
        return loaded
    return None

loaded = None if FORCE_REGENERATE_PARAMS else _load_params()

if loaded is not None:
    mu_list, Sigma_list, alpha, mog_means, mog_variances, weights, x_star = loaded
    mu_list    = [mu.float() for mu in mu_list]
    Sigma_list = [cov.float() for cov in Sigma_list]
    alpha      = alpha.float()
else:
    print("[GMM] Generating fresh parameters...")
    experiment_utils.set_global_seed(GLOBAL_SEED)   # seed generation for reproducibility
    mu_list = [
        torch.tensor([-5,  5], dtype=torch.float64),
        torch.tensor([-5, -5], dtype=torch.float64),
        torch.tensor([ 5,  3], dtype=torch.float64),
        torch.tensor([ 5, -1], dtype=torch.float64),
        torch.tensor([ 0, -3], dtype=torch.float64),
        torch.tensor([-2,  4], dtype=torch.float64),
        torch.tensor([-2, -3], dtype=torch.float64),
        torch.tensor([ 1,  2], dtype=torch.float64),
        torch.tensor([-8,  1], dtype=torch.float64),
        torch.tensor([ 7,  5], dtype=torch.float64),
        torch.tensor([ 0, -5], dtype=torch.float64),
    ]
    Sigma_list = [
        torch.tensor([[0.5000, 0.1950],
                      [0.1950, 0.2000]], dtype=torch.float64)
    ] * len(mu_list)
    alpha = torch.tensor([1 / len(mu_list)] * len(mu_list), dtype=torch.float64)

    mu_list    = [mu.float() for mu in mu_list]
    Sigma_list = [cov.float() for cov in Sigma_list]
    alpha      = alpha.float()

    x_star = torch.tensor([-5])
    mu_temp, Sigma_temp = dist_utils.compute_conditionals(mu_list, Sigma_list, x_star)
    temp_alpha          = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_star)
    mog_means, mog_variances, weights = dist_utils.filter_and_normalize(
        mu_temp, Sigma_temp, temp_alpha, threshold=0.01
    )

    # save to PARAMS_DIR (canonical, share across runs)
    experiment_utils.save_gmm_params(
        mu_list, Sigma_list, alpha,
        mog_means, mog_variances, weights, x_star,
        PARAMS_DIR, EXPERIMENT_NAME
    )

print(f"x_star = {x_star}")
print(f"Number of conditional modes after filtering: {len(mog_means)}")

## Data

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)
X = dist_utils.generate_mog_samples(25_000, mu_list, Sigma_list, alpha).float().to(device)
xh_cpu = X.detach().cpu().numpy()
plt.scatter(xh_cpu[:, 0], xh_cpu[:, 1], alpha=0.6, s=20)
plt.title("Scatter Plot of P(X,Y)")
plt.xlabel("X"); plt.ylabel("Y"); plt.grid(True); plt.show()

## Train Models

### Consistency Model — P(Y|X=x)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

B, C      = X.shape
nfeatures = C - CONDITION_ON
data_generator_cm = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha
)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(
    nfeatures=nfeatures, condition_on=CONDITION_ON,
    nunits=NUNITS_CM, depth=NBLOCKS_CM
)

_loaded_cm = experiment_utils.load_model_checkpoint(
    Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_cm:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    Cos_ConsistencyModeliCT.train_model(
        X=None, nepochs=NEPOCHS_CM, batch_size=BATCH_SIZE_CM,
        device=device, condition=CONDITION_ON,
        data_generator=data_generator_cm, use_improved_training=True
    )
    experiment_utils.save_model_checkpoint(
        Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

### Diffusion — P(Y|X=x)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

X_train   = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X_train.shape[1]

data_generator_diff_cond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha, kernel_func=None
)

model_cond = Diffusion.DiffusionModel(
    nfeatures=nfeatures, nblocks=NBLOCKS, nunits=NUNITS,
    condition=True, condition_on=CONDITION_ON,
    diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_cond = experiment_utils.load_model_checkpoint(
    model_cond, "Diffusion_cond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_cond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_cond.train_model(
        None, data_generator=data_generator_diff_cond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_cond, "Diffusion_cond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

### Diffusion — P(X=x)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

data_generator_diff_uncond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha,
    kernel_func=lambda X: X[:, :CONDITION_ON]
)

model_uncond = Diffusion.DiffusionModel(
    nfeatures=CONDITION_ON, nblocks=NBLOCKS, nunits=NUNITS,
    condition=False, diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_uncond = experiment_utils.load_model_checkpoint(
    model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_uncond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_uncond.train_model(
        None, data_generator=data_generator_diff_uncond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

### SANITY CHECK: Compare CM vs Diffusion conditional quality

In [ ]:
RUN_SANITY_CHECK = True
SANITY_K = 500

if RUN_SANITY_CHECK:
    mmd_loss = MMDLoss(kernel=RBF())
    N_SANITY_SAMPLES = 500

    mmd_diff_list = []
    mmd_cm_list   = []

    for k in trange(SANITY_K, desc="Sanity check"):
        experiment_utils.set_run_seed(GLOBAL_SEED, k)

        # Sample x from the analytic joint distribution, take only the x part
        joint_sample = dist_utils.generate_mog_samples_not_differentiable(
            1, mu_list, Sigma_list, alpha
        ).float()  # shape (1, CONDITION_ON + n_y)
        x_sample = joint_sample[:, :CONDITION_ON]          # shape (1, CONDITION_ON)
        x_vec    = x_sample.view(-1).cpu()                 # shape (CONDITION_ON,)

        # Analytic conditional samples
        mu_cond, Sigma_cond = dist_utils.compute_conditionals(mu_list, Sigma_list, x_vec)
        w_cond = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_vec)
        analytic_samples = dist_utils.generate_mog_samples_not_differentiable(
            N_SANITY_SAMPLES, mu_cond, Sigma_cond, w_cond
        ).float().to(device)

        # Diffusion conditional samples
        cond_rep = x_sample.to(device).repeat(N_SANITY_SAMPLES, 1)
        diff_samples, _, _ = model_cond.sample(
            nsamples=N_SANITY_SAMPLES, condition_x=cond_rep, device=device
        )
        diff_samples = diff_samples[:, CONDITION_ON:]  # keep only y part
        # CM conditional samples
        cm_samples, _, _ = Cos_ConsistencyModeliCT.sample(
            nsamples=N_SANITY_SAMPLES, condition_x=cond_rep, device=device
        )
        # CM already outputs y only (nfeatures = dim_y)

        mmd_diff = mmd_loss(diff_samples, analytic_samples).item()
        mmd_cm   = mmd_loss(cm_samples,   analytic_samples).item()

        mmd_diff_list.append(mmd_diff)
        mmd_cm_list.append(mmd_cm)

    print(f"\n--- Sanity Check Summary (K={SANITY_K}) ---")
    print(f"Diffusion  MMD: mean={np.mean(mmd_diff_list):.5f}  std={np.std(mmd_diff_list):.5f}")
    print(f"CM         MMD: mean={np.mean(mmd_cm_list):.5f}  std={np.std(mmd_cm_list):.5f}")
else:
    print("[Sanity check skipped] Set RUN_SANITY_CHECK = True to run.")

## Optimize

### LGD

In [ ]:
# NOTE: optimize_LGD call is untouched — only seed management added around it
best_x_t_LGD_list = []
l2_gmm_LGD_list   = []
l2_x_LGD_list     = []
lgd_times         = []
final_loss_LGD    = []

for i in trange(N_ATTEMP_OPTIM):
    run_seed = experiment_utils.set_run_seed(GLOBAL_SEED, i)

    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, model_cond, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, loss="MMD", device=device,
        num_x_t=NUM_X_T_LGD
    )
    end_time = time.time()

    lgd_times.append(end_time - start_time)
    final_loss_LGD.append(final_loss)
    best_x_t_LGD_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = dist_utils.gmm_l2_distance(
        mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights
    )
    l2_x = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_list.append(l2_gmm)
    l2_x_LGD_list.append(l2_x)
    print(f"[{i+1}] seed={run_seed} | L2 GMM: {l2_gmm:.6f} | L2 to x*: {l2_x:.6f}")

### LGD-CM

In [ ]:
# NOTE: optimize_LGD call is untouched — only seed management added around it
best_x_t_LGD_CM_list = []
l2_gmm_LGD_CM_list   = []
l2_x_LGD_CM_list     = []
lgd_cm_times         = []
final_loss_LGD_CM    = []

for i in trange(N_ATTEMP_OPTIM):
    run_seed = experiment_utils.set_run_seed(GLOBAL_SEED, i)

    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, Cos_ConsistencyModeliCT,
        mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, loss="MMD", device=device,
        CM=True, FLAG=False, num_x_t=NUM_X_T_LGD_CM
    )
    end_time = time.time()

    lgd_cm_times.append(end_time - start_time)
    final_loss_LGD_CM.append(final_loss)
    best_x_t_LGD_CM_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = dist_utils.gmm_l2_distance(
        mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights
    )
    l2_x = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_CM_list.append(l2_gmm)
    l2_x_LGD_CM_list.append(l2_x)
    print(f"[{i+1}] seed={run_seed} | L2 GMM: {l2_gmm:.6f} | L2 to x*: {l2_x:.6f}")

## Results

In [ ]:
rows = [
    experiment_utils.summary_row("LGD",    l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    experiment_utils.summary_row("LGD-CM", l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
]
df = pd.DataFrame(rows).set_index("Method")
display(df)

rows_top10 = [
    experiment_utils.top10_stats("LGD",    final_loss_LGD,    l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    experiment_utils.top10_stats("LGD-CM", final_loss_LGD_CM, l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
]
df_top10 = pd.DataFrame(rows_top10).set_index("Method")
display(df_top10)

In [ ]:
from google.colab import files
import zipfile

# 1. Download the Results JSON
print(f"Downloading results: {path}")
files.download(path)

# 2. Zip the Checkpoints directory and download it
zip_path = f"/content/{EXPERIMENT_NAME}_checkpoints.zip"
print(f"Zipping checkpoints to {zip_path}...")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk(CHECKPOINT_DIR):
        for file in files_in_dir:
            file_full_path = os.path.join(root, file)
            # Store with a relative path inside the zip
            arcname = os.path.relpath(file_full_path, CHECKPOINT_DIR)
            zipf.write(file_full_path, arcname)

print("Downloading checkpoints zip...")
files.download(zip_path)

In [ ]:
def to_python(val):
    if isinstance(val, torch.Tensor):
        return val.detach().cpu().tolist()
    if isinstance(val, np.ndarray):
        return val.tolist()
    if hasattr(val, "item"):
        return val.item()
    return val

results = {
    "experiment":  EXPERIMENT_NAME,
    "seed":        GLOBAL_SEED,
    "environment": env_info,
    "LGD": {
        "x_pred":     [to_python(x) for x in best_x_t_LGD_list],
        "final_loss": [to_python(l) for l in final_loss_LGD],
        "l2_gmm":     l2_gmm_LGD_list,
        "l2_x":       l2_x_LGD_list,
        "times":      lgd_times,
    },
    "LGD-CM": {
        "x_pred":     [to_python(x) for x in best_x_t_LGD_CM_list],
        "final_loss": [to_python(l) for l in final_loss_LGD_CM],
        "l2_gmm":     l2_gmm_LGD_CM_list,
        "l2_x":       l2_x_LGD_CM_list,
        "times":      lgd_cm_times,
    },
    "meta": {
        "n_attemp_optim":            N_ATTEMP_OPTIM,
        "nsamples_in_optim_for_mmd": NSAMPLES_IN_OPTIM_FOR_MMD,
        "x_star":                    to_python(x_star),
    },
}

path = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_results_seed{GLOBAL_SEED}.json")
with open(path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {path}")

# CDMS: Sampling vs Analytical Q(x) across ζ values

In [ ]:
import os
import math
import json
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from functools import partial
from tqdm import tqdm, trange
from scipy.stats import gaussian_kde

In [ ]:
BATCH_SIZE_CM=2*4096
NEPOCHS_CM=80_000

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

B, C      = X.shape
nfeatures = C - CONDITION_ON
data_generator_cm = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha
)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(
    nfeatures=nfeatures, condition_on=CONDITION_ON,
    nunits=NUNITS_CM, depth=NBLOCKS_CM
)

_loaded_cm = experiment_utils.load_model_checkpoint(
    Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_cm:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    Cos_ConsistencyModeliCT.train_model(
        X=None, nepochs=NEPOCHS_CM, batch_size=BATCH_SIZE_CM,
        device=device, condition=CONDITION_ON,
        data_generator=data_generator_cm, use_improved_training=True
    )
    experiment_utils.save_model_checkpoint(
        Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

In [ ]:
# ============================================================
# CELL 5.5: Helper functions for grid evaluation
# ============================================================

def marginal_density(x_val):
    """
    Evaluate the marginal density P(x) at a scalar x_val
    by marginalising the joint GMM over Y.
    """
    x_tensor = torch.tensor([x_val], dtype=torch.float32)
    density = torch.zeros(1)
    for k, (mu_k, Sigma_k, w_k) in enumerate(zip(mu_list, Sigma_list, alpha)):
        # marginal over X is the first dimension of the joint GMM
        mu_x    = mu_k[0]                   # scalar
        Sigma_x = Sigma_k[0, 0]             # scalar variance
        density += w_k * torch.exp(
            -0.5 * ((x_tensor - mu_x) ** 2) / Sigma_x
        ) / (2 * torch.pi * Sigma_x).sqrt()
    return density.item()

def l2gmm_loss_at_x(x_val):
    """
    Evaluate the GMM L2 loss L(x) at a scalar x_val.
    Uses the closed-form gmm_l2_diff between P(Y|X=x) and G(Y).
    """
    x_tensor = torch.tensor([x_val], dtype=torch.float32)

    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_tensor)
    w_pred              = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_tensor)

    # mu_pred: (K1, D_y, 1) -> (K1, D_y)
    mu_pred_d    = mu_pred.squeeze(-1)

    # Sigma_pred: ensure (K1, D_y, D_y) — guard against squeezed dims
    Sigma_pred_d = Sigma_pred
    if Sigma_pred_d.dim() == 2:
        Sigma_pred_d = Sigma_pred_d.unsqueeze(-1)          # (K1, 1) -> (K1, 1, 1)
    if Sigma_pred_d.dim() == 1:
        Sigma_pred_d = Sigma_pred_d.unsqueeze(-1).unsqueeze(-1)

    w_pred_d = w_pred

    # mog_means: (K2, 1, D_y) -> (K2, D_y)
    mog_means_d = mog_means.squeeze(1)

    # mog_variances: ensure (K2, D_y, D_y)
    mog_vars_d = mog_variances.squeeze(1)
    if mog_vars_d.dim() == 2:
        mog_vars_d = mog_vars_d.unsqueeze(-1)              # (K2, 1) -> (K2, 1, 1)
    if mog_vars_d.dim() == 1:
        mog_vars_d = mog_vars_d.unsqueeze(-1).unsqueeze(-1)

    weights_d = weights

    with torch.no_grad():
        loss = gmm_l2_diff(
            mu_pred_d, Sigma_pred_d, w_pred_d,
            mog_means_d, mog_vars_d, weights_d
        )
    return loss.item()

In [ ]:
def compute_l2_gmm_loss(x0_sample, mu_list, Sigma_list, alpha,
                        mog_means, mog_variances, weights, device):
    """
    Compute differentiable GMM L2 loss between P(Y|X=x0_sample) and G(Y).
    Gradient flows back through x0_sample -> x_t.

    Args:
        x0_sample:     (1, nfeatures) tensor, requires_grad, on device
        mu_list:       list of (D,) tensors — joint GMM means
        Sigma_list:    list of (D, D) tensors — joint GMM covariances
        alpha:         (K,) tensor — joint GMM weights
        mog_means:     (K2, 1, D_y) tensor — target conditional means
        mog_variances: (K2, 1, D_y, D_y) tensor — target conditional covariances
        weights:       (K2,) tensor — target conditional weights
        device:        torch device
    """
    x0_cpu = x0_sample.view(-1).cpu()

    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x0_cpu)
    w_pred              = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x0_cpu)

    mu_pred_d    = mu_pred.squeeze(-1).to(device)      # (K1, D_y)
    Sigma_pred_d = Sigma_pred.to(device)               # (K1, D_y, D_y)
    w_pred_d     = w_pred.to(device)                   # (K1,)

    mog_means_d  = mog_means.squeeze(1).to(device)     # (K2, D_y)
    mog_vars_d   = mog_variances.squeeze(1).to(device) # (K2, D_y, D_y)
    weights_d    = weights.to(device)                  # (K2,)

    return gmm_l2_diff(mu_pred_d, Sigma_pred_d, w_pred_d,
                       mog_means_d, mog_vars_d, weights_d)


In [ ]:
def optimize_DPS(model_uncond, model_cond, mog_means, mog_variances, weights,
                 mu_list, Sigma_list, alpha,
                 nsamples=250, loss="MMD", CM=False,
                 device="cuda", FLAG=False, zeta=1.0):

    mmd_loss = MMDLoss(kernel=RBF())

    x_t = torch.randn((1, model_uncond.nfeatures), device=device, requires_grad=True)

    pbar = tqdm(range(model_uncond.diffusion_steps - 1, 0, -1)) if FLAG \
           else range(model_uncond.diffusion_steps - 1, 0, -1)

    for t in pbar:
        x_t = x_t.detach().clone().requires_grad_(True)

        x_t_minus_1, pred_x0 = model_uncond.sample_ddim_step(
            x_t, t, condition_x=None, device=device, eta=1.0
        )

        if zeta == 0.0:
            with torch.no_grad():
                x_t = x_t_minus_1.detach().clone()
            continue

        # DPS: single estimate directly on pred_x0, no sampling noise
        if loss == "L2_GMM":
            loss_val = compute_l2_gmm_loss(
                pred_x0, mu_list, Sigma_list, alpha,
                mog_means, mog_variances, weights, device
            )
        else:
            condition = pred_x0.view(1, -1).repeat(nsamples, 1)
            target_samples, _, _ = model_cond.sample(
                nsamples=nsamples, condition_x=condition, device=device
            )
            if not CM:
                target_samples = target_samples[:, model_cond.condition_on:]
            mog_samples = dist_utils.generate_mog_samples_not_differentiable(
                nsamples, mog_means, mog_variances, weights
            )
            loss_val = mmd_loss(target_samples, mog_samples)

        if FLAG:
            pbar.set_description(
                f"Step {t} | loss:{loss_val.item():.4f} | pred_x0:{pred_x0.item():.4f}"
            )

        loss_val.backward()

        grad = x_t.grad.clone()
        grad = torch.clamp(grad, min=-0.5, max=0.5)

        with torch.no_grad():
            x_t = x_t_minus_1.detach().clone() - (zeta * grad)

    # Final loss evaluation
    condition = x_t.view(1, -1).repeat(nsamples, 1)
    target_samples, _, _ = model_cond.sample(
        nsamples=nsamples, condition_x=condition, device=device
    )
    if not CM:
        target_samples = target_samples[:, model_cond.condition_on:]
    mog_samples = dist_utils.generate_mog_samples_not_differentiable(
        nsamples, mog_means, mog_variances, weights
    )
    final_loss = mmd_loss(target_samples, mog_samples)

    x_t_final = x_t.detach().clone()
    del x_t, condition, target_samples, mog_samples
    torch.cuda.empty_cache() if device == "cuda" else None

    return x_t_final, x_t_final, final_loss.detach()

In [ ]:
ZETA_VALUES = [0.0, .25, 2., 4., 8., 16.]

print("Computing P(x) and L(x) on grid ...")
x_grid  = np.linspace(-8, 8, 300)                          # FIX 1: moved here
px_grid = np.array([marginal_density(xv) for xv in x_grid])
lx_l2   = np.array([l2gmm_loss_at_x(xv)  for xv in x_grid])
lx_grid = lx_l2                                            # FIX 2: alias assigned

analytical_Q = {}                                          # FIX 3: computed once
for zeta in ZETA_VALUES:
    q_unnorm           = px_grid * np.exp(-zeta * lx_grid)
    Z                  = np.trapz(q_unnorm, x_grid)
    analytical_Q[zeta] = q_unnorm / Z

In [ ]:
# ============================================================
# Plot L(x) and analytical Q(x; beta) for each zeta
# ============================================================
fig, axes = plt.subplots(1, len(ZETA_VALUES) + 1, figsize=(4 * (len(ZETA_VALUES) + 1), 3))

# Left panel: L(x)
axes[0].plot(x_grid, lx_l2, color='darkorange', lw=2)
axes[0].axvline(x_star.item(), color='k', ls=':', lw=1.5, label=f"x*={x_star.item():.1f}")
axes[0].set_title(r"$\mathcal{L}(x)$ — L2-GMM")
axes[0].set_xlabel("$x$")
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

# One panel per zeta: analytical Q(x; beta)
for ax, zeta in zip(axes[1:], ZETA_VALUES):
    ax.plot(x_grid, analytical_Q[zeta], color='darkorange', lw=2)
    ax.fill_between(x_grid, analytical_Q[zeta], alpha=0.2, color='darkorange')
    ax.axvline(x_star.item(), color='k', ls=':', lw=1.5, label=f"x*={x_star.item():.1f}")
    ax.set_title(rf"$\mathcal{{Q}}_\beta$,  $\beta={zeta}$")
    ax.set_xlabel("$x$")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle(
    r"$\mathcal{Q}_\beta(x) \propto \mathcal{P}(x)\,e^{-\beta\,\mathcal{L}(x)}$"
    r" — prior ($\beta=0$) $\to$ optimization ($\beta\to\infty$)",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_analytical_Q.pdf"),
            bbox_inches='tight')
plt.show()
print("Analytical Q plotted.")

In [ ]:

# ============================================================
# CELL 7: CDMS sampling  (FIX 4: N=10 block removed)
# ============================================================
N_CDMS_SAMPLES = 10

cdms_samples = {}

for zeta in ZETA_VALUES:
    print(f"\n[CDMS] Sampling β = {zeta} ...")
    samples = []
    for i in trange(N_CDMS_SAMPLES):
        experiment_utils.set_run_seed(GLOBAL_SEED, i)
        x_pred, _, _ = optimize_DPS(
            model_uncond, Cos_ConsistencyModeliCT,
            mog_means, mog_variances, weights,
            mu_list, Sigma_list, alpha,
            nsamples=NSAMPLES_IN_OPTIM_FOR_MMD,
            loss="MMD", device=device,
            CM=True, FLAG=False,
            zeta=zeta
        )
        print(x_pred)
        samples.append(x_pred.float().view(-1).cpu()[0].item())
    cdms_samples[zeta] = np.array(samples)
    print(f"   mean={np.mean(samples):.3f} | std={np.std(samples):.3f}")

In [ ]:

# ============================================================
# CELL 8: Save CDMS samples to JSON
# ============================================================
cdms_output = {
    f"zeta_{zeta}": {"zeta": zeta, "samples": cdms_samples[zeta].tolist()}
    for zeta in ZETA_VALUES
}

save_path = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_CDMS_samples.json")
with open(save_path, "w") as f:
    json.dump(cdms_output, f, indent=2)

print(f"Saved to {save_path}")

try:
    from google.colab import files
    files.download(save_path)
except ImportError:
    pass  # not in Colab



In [ ]:

# ============================================================
# CELL 9: 2x3 beta-sweep plot
# ============================================================
plt.rcParams.update({
    'font.size': 9, 'axes.titlesize': 10,
    'axes.labelsize': 8, 'xtick.labelsize': 7,
    'ytick.labelsize': 7, 'legend.fontsize': 7,
})

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
axes_flat = axes.flatten()

for ax, zeta in zip(axes_flat, ZETA_VALUES):
    samples = cdms_samples[zeta]

    ax.plot(x_grid, analytical_Q[zeta],
            color="#E53935", lw=2, linestyle='--',
            label=r"Analytical $\mathcal{Q}_\beta$")
    ax.fill_between(x_grid, analytical_Q[zeta], alpha=0.12, color="#E53935")

    if len(samples) > 1 and np.std(samples) > 1e-6:
        ax.hist(samples, bins=50, density=True,
                color="#1E88E5", alpha=0.6, edgecolor="white",
                label=r"Sampled $x$")
    else:
        ax.axvline(samples[0], color="#1E88E5", lw=2, label=r"Sampled $x$")

    ax.axvline(x_star.item(), color='k', linestyle=':', lw=1.5,
               label=f"$x^*={x_star.item()}$")
    ax.set_title(rf"$\beta = {zeta}$", fontsize=14)
    ax.set_xlabel("$x$"); ax.set_ylabel("Density")
    ax.set_xlim(x_grid[0], x_grid[-1])
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle(
    r"CDMS: $\mathcal{Q}_\beta(x) \propto \mathcal{P}(x)\,e^{-\beta\,\mathcal{L}(x)}$"
    r" — prior ($\beta=0$) $\to$ optimization ($\beta\to\infty$)",
    fontsize=13
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_CDMS_beta_sweep.pdf"),
            bbox_inches='tight')
plt.show()
print("CDMS beta-sweep plot saved.")



In [ ]:

# ============================================================
# CELL 10: Final 2x2 summary figure
# ============================================================
results_path = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_results_seed{GLOBAL_SEED}.json")
cdms_path    = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_CDMS_samples.json")

with open(results_path, 'r') as f:
    results = json.load(f)
with open(cdms_path, 'r') as f:
    cdms_raw = json.load(f)

cdms_samples = {float(v['zeta']): np.array(v['samples']) for k, v in cdms_raw.items()}

l2_gmm_LGD_list      = results['LGD']['l2_gmm']
lgd_times            = results['LGD']['times']
best_x_t_LGD_list    = [torch.tensor(x) for x in results['LGD']['x_pred']]

l2_gmm_LGD_CM_list   = results['LGD-CM']['l2_gmm']
lgd_cm_times         = results['LGD-CM']['times']
best_x_t_LGD_CM_list = [torch.tensor(x) for x in results['LGD-CM']['x_pred']]

x_star = torch.tensor(results['meta']['x_star'])

best_idx_lgd    = int(np.argmin(l2_gmm_LGD_list))
best_idx_lgd_cm = int(np.argmin(l2_gmm_LGD_CM_list))
best_x_lgd      = best_x_t_LGD_list[best_idx_lgd].float().view(-1).cpu()
best_x_lgd_cm   = best_x_t_LGD_CM_list[best_idx_lgd_cm].float().view(-1).cpu()

y_grid = torch.linspace(-8, 8, 500)

def cond_pdf(x_val):
    mu_c, Sig_c = dist_utils.compute_conditionals(mu_list, Sigma_list, x_val)
    w_c         = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_val)
    pdf = sum(
        w * torch.exp(-0.5 * ((y_grid - m) ** 2) / s) / (2 * torch.pi * s).sqrt()
        for w, m, s in zip(w_c, mu_c, Sig_c)
    )
    return pdf.numpy().squeeze()

pdf_gt     = cond_pdf(x_star.float())
pdf_lgd    = cond_pdf(best_x_lgd)
pdf_lgd_cm = cond_pdf(best_x_lgd_cm)

zeta_values = [k for k in cdms_samples.keys() if k in (0, 0.0, 4, 4.0)]

legend_kw = dict(fontsize=7, framealpha=0.9, handlelength=1.2,
                 borderpad=0.4, labelspacing=0.3,
                 handletextpad=0.4, borderaxespad=0.4)

fig, axes = plt.subplots(2, 2, figsize=(10, 6))

# Top-left: joint scatter
ax = axes[0, 0]
xh_cpu = X.detach().cpu().numpy()
ax.scatter(xh_cpu[:, 0], xh_cpu[:, 1], alpha=0.4, s=4, color='steelblue')
ax.axvline(x_star.item(),        color='k',       linestyle='--', lw=1.2,
           label=f"$x^*={x_star.item()}$")
ax.axvline(best_x_lgd.item(),    color='#E53935', linestyle='-',  lw=1.2,
           label=rf"LGD $\hat{{x}}^*={best_x_lgd.item():.3f}$")
ax.axvline(best_x_lgd_cm.item(), color='#43A047', linestyle='-',  lw=1.2,
           label=rf"MLGD-D $\hat{{x}}^*={best_x_lgd_cm.item():.3f}$")
ax.set_title(r"$\mathcal{P}(X, Y)$")
ax.set_xlabel("$X$"); ax.set_ylabel("$Y$")
ax.legend(**legend_kw); ax.grid(True, alpha=0.3)

# Bottom-left: conditional PDF
ax = axes[1, 0]
y_np = y_grid.numpy()
ax.plot(y_np, pdf_gt,     color='k',       linestyle='--', lw=1.5,
        label=r"$\mathcal{G}(Y)$")
ax.plot(y_np, pdf_lgd,    color='#E53935', linestyle='-',  lw=1.2,
        label=rf"LGD $\hat{{x}}^*={best_x_lgd.item():.3f}$, "
              rf"$L^2={l2_gmm_LGD_list[best_idx_lgd]:.4f}$")
ax.plot(y_np, pdf_lgd_cm, color='#43A047', linestyle='-',  lw=1.2,
        label=rf"MLGD-D $\hat{{x}}^*={best_x_lgd_cm.item():.3f}$, "
              rf"$L^2={l2_gmm_LGD_CM_list[best_idx_lgd_cm]:.4f}$")
ax.set_title(r"$\mathcal{P}(Y \mid X = \hat{x}^*)$")
ax.set_xlabel("$y$"); ax.set_ylabel("Density")
ax.legend(**legend_kw); ax.grid(True, alpha=0.3)

# Right column: CDMS β=0 and β=4
right_titles = [
    r"$\mathcal{Q}_\beta(x)$,  $\beta=0$ (prior)",
    r"$\mathcal{Q}_\beta(x)$,  $\beta=4$ (guided)",
]
for ax, zeta, title in zip([axes[0, 1], axes[1, 1]], zeta_values, right_titles):
    samples = np.atleast_1d(cdms_samples[zeta])

    ax.plot(x_grid, analytical_Q[zeta],
            color="#E53935", lw=1.5, linestyle='--',
            label=r"Analytical $\mathcal{Q}_\beta$")
    ax.fill_between(x_grid, analytical_Q[zeta], alpha=0.12, color="#E53935")

    if len(samples) > 1 and np.std(samples) > 1e-6:
        ax.hist(samples, bins=50, density=True,
                color="#1E88E5", alpha=0.6, edgecolor="white",
                label=r"Sampled $x$")
    else:
        ax.axvline(samples[0], color="#1E88E5", lw=1.5, label=r"Sampled $x$")

    ax.axvline(x_star.item(), color='k', linestyle=':', lw=1.2,
               label=f"$x^*={x_star.item()}$")
    ax.set_title(title)
    ax.set_xlabel("$x$"); ax.set_ylabel("Density")
    ax.set_xlim(x_grid[0], x_grid[-1])
    ax.legend(**legend_kw); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_2DCOND1D.pdf"),
            bbox_inches='tight', dpi=300)
plt.show()
print("Final 2x2 plot saved.")